# 05 Memorizing noise: random labels, weight decay, early stopping

Lecture 10, training and generalization. Book chapter 7 (memorization and
double descent) and chapter 6 (regularization).

**Goals**

- Train the same MLP on true labels and on *random* labels (Zhang et al.
  2017), and on labels with a growing fraction of corruption.
- See how weight decay and early stopping change what the network fits.

**Expected findings** (asserted in `tests/test_examples.py::test_05_*`):

1. The network reaches 100 % training accuracy on random labels, with test
   accuracy at chance; random labels take longer to fit than true labels.
2. Strong weight decay prevents the memorization of random labels but still
   lets the network learn the true labels.
3. With 30 % flipped labels, validation accuracy peaks early and then drops
   while the network memorizes the flipped labels: early stopping helps.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import glassnn.functional as F
from glassnn import Tensor, manual_seed, nn, no_grad, optim
from glassnn.data import DataLoader

D, N = 20, 200
rng = np.random.default_rng(0)
w_star = rng.normal(size=D)


def sample(n):
    # Gaussian inputs; the true label is the side of a hyperplane.
    X = rng.normal(size=(n, D))
    return X, (X @ w_star > 0).astype(np.int64)


X_train, y_train = sample(N)
X_test, y_test = sample(2000)
y_random = rng.integers(0, 2, size=N)

In [ ]:
def accuracy(model, X, y):
    model.eval()
    with no_grad():
        value = float((model(Tensor(X)).data.argmax(axis=1) == y).mean())
    model.train()
    return value


def train(labels, epochs, weight_decay=0.0, X_val=X_test, y_val=y_test):
    # Return the (train accuracy, validation accuracy) after every epoch.
    manual_seed(0)
    model = nn.Sequential(
        nn.Linear(D, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 2)
    )
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    loader = DataLoader(X_train, labels, batch_size=32, shuffle=True)
    history = []
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            optimizer.step()
        history.append(
            (accuracy(model, X_train, labels), accuracy(model, X_val, y_val))
        )
    return np.array(history)

## 1. True labels against random labels

The network has about 72 000 parameters for 200 training points.

In [ ]:
true = train(y_train, epochs=40)
random = train(y_random, epochs=40)

fig, ax = plt.subplots(figsize=(6, 4))
for history, name in [(true, "true labels"), (random, "random labels")]:
    ax.plot(history[:, 0], label=f"{name}: train")
    ax.plot(history[:, 1], "--", label=f"{name}: test")
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.legend()
plt.show()


def first_perfect_epoch(history):
    return int(np.argmax(history[:, 0] == 1.0))


print(
    "epochs to 100 % training accuracy:",
    "true",
    first_perfect_epoch(true),
    "| random",
    first_perfect_epoch(random),
)
print("final test accuracy: true", true[-1, 1], "| random", random[-1, 1])

The same architecture, optimizer and number of steps can fit labels that
carry no information at all. Whatever makes the network generalize on the
true labels, it is not a restriction of the hypothesis class that the
optimizer explores: its empirical Rademacher complexity on this sample is
essentially maximal, and a uniform bound based on it is vacuous (lecture 2).

## 2. Partially corrupted labels

A fraction $q$ of the labels is replaced by random labels.

In [ ]:
corruption_rng = np.random.default_rng(1)
fractions = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
epochs_to_fit, test_accuracy = [], []
for q in fractions:
    runs = []  # three draws of the corrupted labels
    for _ in range(3):
        corrupted = corruption_rng.uniform(size=N) < q
        labels = np.where(corrupted, corruption_rng.integers(0, 2, size=N), y_train)
        history = train(labels, epochs=80)
        runs.append((first_perfect_epoch(history), history[-1, 1]))
    epochs_to_fit.append(np.mean([r[0] for r in runs]))
    test_accuracy.append(np.mean([r[1] for r in runs]))
    print(
        f"q = {q:.1f}: epochs to fit {epochs_to_fit[-1]:4.1f}, "
        f"test accuracy {test_accuracy[-1]:.3f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(fractions, epochs_to_fit, "o-")
axes[0].set_ylabel("epochs to 100 % training accuracy")
axes[1].plot(fractions, test_accuracy, "o-")
axes[1].set_ylabel("test accuracy (mean of 3 draws)")
for ax in axes:
    ax.set_xlabel("fraction of corrupted labels $q$")
plt.show()

## 3. Weight decay

AdamW with decoupled weight decay $\lambda$ shrinks every weight by the factor
$1 - \eta\lambda$ at every step (chapter 5). Final accuracies after 100
epochs:

In [ ]:
for weight_decay in [0.0, 1.0, 20.0, 50.0]:
    on_random = train(y_random, epochs=100, weight_decay=weight_decay)[-1]
    on_true = train(y_train, epochs=100, weight_decay=weight_decay)[-1]
    print(
        f"weight decay {weight_decay:5.1f}: "
        f"random labels train {on_random[0]:.3f} | "
        f"true labels train {on_true[0]:.3f}, test {on_true[1]:.3f}"
    )

Moderate weight decay changes almost nothing: the random labels are still
memorized. This is the observation of Zhang et al. (2017) that explicit
regularization is neither necessary nor sufficient for generalization. Very
strong weight decay ($\lambda = 50$) does prevent memorization, while the
true labels, which a small-norm network can explain, are still learned.

## 4. Early stopping with noisy labels

30 % of the training labels are flipped; the validation set has clean labels.

In [ ]:
flipped = rng.uniform(size=N) < 0.3
y_noisy = np.where(flipped, 1 - y_train, y_train)
X_val, y_val = sample(1000)
history = train(y_noisy, epochs=100, X_val=X_val, y_val=y_val)
best = int(np.argmax(history[:, 1]))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history[:, 0], label="train (noisy labels)")
ax.plot(history[:, 1], label="validation (clean labels)")
ax.axvline(best, color="gray", linestyle=":", label=f"best epoch ({best})")
ax.set_xlabel("epoch")
ax.set_ylabel("accuracy")
ax.legend()
plt.show()
print(
    f"best validation accuracy {history[best, 1]:.3f} at epoch {best} "
    f"(train {history[best, 0]:.3f}); "
    f"final {history[-1, 1]:.3f} (train {history[-1, 0]:.3f})"
)

**Findings.**

1. Random labels are memorized perfectly, about three times more slowly than
   true labels, with test accuracy at chance. With partially corrupted
   labels the test accuracy falls steadily with $q$; the time to fit jumps
   as soon as some labels are corrupted (here it hardly depends on $q$
   beyond that: 200 points are few for this network).
2. Only very strong weight decay prevents memorization; it keeps the signal.
3. The network first learns the structure shared by most samples, then
   memorizes the flipped labels; validation accuracy is best before the
   training accuracy reaches 100 %, so early stopping helps.

**Exercises.**

1. Replace the inputs of the random-label experiment by pure noise that is
   not shared with the test set. Does fitting take longer?
2. Add `nn.Dropout(0.5)` after each ReLU. Does it prevent memorization of
   random labels? Of noisy labels?